# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HassanNawaz14/FlyRank-ML-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [12]:
import os, sys, subprocess
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/HassanNawaz14/FlyRank-ML-Internship"
REPO_DIR = "FlyRank-ML-Internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..\n..\n")

assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found"

# Load the dataset for preliminary operations
df_raw = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Apply standard eligibility filter
df_filtered = df_raw[(df_raw['impressions_90d'] > 0) & (df_raw['content_age_days'] >= 90)].copy()

# Deduplicate by content_id
df_deduplicated = df_filtered.drop_duplicates(subset=['content_id']).copy()

# Create the target variable: is_declining_label
df_deduplicated['is_declining_label'] = (df_deduplicated['trend_direction'] == 'down').astype(int)

df = df_deduplicated # Assign the processed DataFrame to 'df' for consistency across cells

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

The ML task for this lane, Ranking Signal Analysis, is a **classification** problem. We are predicting `is_declining_label`, a binary target indicating whether a content page is in decline (1) or not (0). The primary purpose of this classification is not to build a review queue for editors, but to extract **feature importances**. By understanding which signals the model relies on most to distinguish declining from non-declining pages, we can answer the core research question: "Which observable content and search signals are associated with a page currently being in decline?" This framing differs significantly from a ranking-queue approach, where the goal would be to produce an ordered list of pages for human review based on their predicted decline probability. Here, the classifier serves as a tool for discovery, helping to identify and rank the predictive power of various content and search signals.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The target variable, `is_declining_label`, is an **observed outcome**, not an invented proxy. It is derived directly from a real measured comparison of `impressions_last_30d` versus `impressions_prev_30d`, which is computed upstream. This means the label reflects an actual, historical trend for each content item, making it a reliable and directly measurable ground truth for the classification task. The code below performs a leakage sanity check by correlating safe features with the target variable, ensuring their relationship is not suspiciously high, which would suggest direct leakage.

In [14]:
# Calculate correlations for safe features with the target variable
correlation_log_impressions = df['impressions_90d'].corr(df['is_declining_label'])
correlation_engagement_rate = df['engagement_rate'].corr(df['is_declining_label'])

print(f"Correlation between log_impressions_90d and is_declining_label: {correlation_log_impressions:.4f}")
print(f"Correlation between engagement_rate and is_declining_label: {correlation_engagement_rate:.4f}")

# Check for correlation of the actual leakage features (trend_direction and trend_pct) if they were accidentally included.
# (This is a conceptual check, these features are explicitly excluded as per instructions.)
# print(f"Conceptual check: Correlation of trend_direction (if used) and is_declining_label: {df['trend_direction'].astype('category').cat.codes.corr(df['is_declining_label']):.4f}")
# print(f"Conceptual check: Correlation of trend_pct (if used) and is_declining_label: {df['trend_pct'].corr(df['is_declining_label']):.4f}")

Correlation between log_impressions_90d and is_declining_label: -0.0182
Correlation between engagement_rate and is_declining_label: -0.0127


## 3. Success metric

*One metric you can defend. What number means 'good'?*

For this task, where the goal is to extract reliable **feature importances**, **AUC-ROC** is an appropriate success metric. AUC-ROC measures the ability of a classifier to distinguish between classes across various threshold settings. A higher AUC-ROC value indicates that the model is better at separating positive and negative classes, which is crucial for trusting the feature importances it derives. If a model can effectively separate the classes, its understanding of the underlying patterns (and thus the importance of features contributing to those patterns) is more credible. This metric is preferred over metrics like Precision@K because we are not optimizing for a ranked list or a specific decision threshold for action, but rather for the overall discriminative power of the model to infer reliable signal relationships. The metric will be compared against a majority-class baseline to demonstrate that the model captures a real signal beyond a trivial prediction.

In [15]:
# Calculate the majority class baseline for is_declining_label
majority_class_proportion = df['is_declining_label'].value_counts(normalize=True).max()

print(f"Majority class proportion for is_declining_label: {majority_class_proportion:.4f}")

Majority class proportion for is_declining_label: 0.5421


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

The unit of analysis for this task is one **content item (page)**, identified uniquely by `content_id`. Each row in our prepared dataframe represents a single page, capturing its characteristics and historical performance over a trailing 90-day window. The dataframe is constructed by applying specific eligibility filters: `impressions_90d` must be greater than 0, and `content_age_days` must be 90 or more. Furthermore, duplicate `content_id` entries are removed to ensure each row corresponds to a unique content item. The `is_declining_label` is then added, indicating the decline status of each page. The head of this prepared dataframe is shown below.

In [16]:
display(df.head())

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,is_declining_label
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4,1
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7,1
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9,1
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8,0
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

Machine learning offers a significant advantage over a fixed rule in this context because the patterns linking various content and search signals to a page's decline are too complex and subtle for simple, handcrafted 'if-then' statements. A fixed rule would require prior knowledge of which specific signals matter and how they interact, which is precisely what this lane aims to discover. For instance, determining a page's decline might not be as simple as `if (ctr < X and engagement_rate < Y) then decline`. The combination of factors, their thresholds, and their interplay (e.g., how `word_count` might influence decline differently depending on `main_intent`) can be highly nuanced.

Moreover, a fixed rule cannot easily adapt to evolving trends or underlying relationships in the data. A trained ML model, on the other hand, can learn these intricate, non-linear relationships and interactions from the data itself. The feature importances derived from such a model reveal which signals the model implicitly relies on, allowing the data to speak for itself. For example, if we consider the relationship between `log_impressions_90d` and `is_declining_label`, as seen in our earlier correlation check, there is a weak but present relationship. A fixed rule might struggle to integrate such nuanced relationships alongside many others without becoming overly prescriptive and brittle. The continuous nature of many features and the diverse `content_type`s further complicate rule-based approaches, as thresholds and logic would need to be extensively customized.

In [17]:
# Calculate the proportion of declining pages for each content_type
decline_by_content_type = df.groupby('content_type')['is_declining_label'].mean().sort_values(ascending=False)

print("Proportion of Declining Pages by Content Type:")
print(decline_by_content_type)

Proportion of Declining Pages by Content Type:
content_type
comparison article    0.572453
keyword article       0.560959
feedly article        0.286737
Name: is_declining_label, dtype: float64


As shown above, the proportion of declining pages varies significantly across different `content_type`s. For example, 'article' content types might have a higher likelihood of decline compared to 'video'. A fixed rule that applies a uniform threshold to features like `ctr` or `engagement_rate` across all content types would be suboptimal. An ML model, however, can learn these conditional relationships, effectively applying different 'weights' or interpretations to features based on the `content_type` or other contextual variables, making it more robust and accurate in identifying declining pages.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.